**Tasks:**
- For simplicity, we are going to focus on analyzing scores for Men's Free Skate and Women's Free Skate for each competition in ISU Grand Prix for the 2024/2025 season
- In this deliverable, I will focus on scraping the data from all the pdfs that fit into these categories into their each csv

In [5]:
import pandas as pd
import numpy as np
import pdfplumber
import re

In [19]:
# 2024 Allen Mens FS

text = ""

with pdfplumber.open('2024_AllenComp_MenFS.pdf') as pdf:
    for page in pdf.pages:
        text += page.extract_text() + "\n"

skater_blocks = text.split("Rank Name Nation")

all_elements = []

for block in skater_blocks[1:]:
    
    # --- Extract skater name ---
    name_match = re.search(r"\d+\s+([A-Za-z\s\-']+)\s+[A-Z]{3}", block)
    if not name_match:
        continue
    
    skater_name = name_match.group(1).strip()
    
    # --- Extract elements ---
    element_pattern = re.compile(
        r"\n\s*(\d+)\s+([A-Za-z0-9\+\*q<>]+)\s+([\d\.]+)\s+([\d\.\-]+).*?([\d\.]+)$",
        re.MULTILINE
    )
    
    for match in element_pattern.finditer(block):
        all_elements.append({
            "Skater": skater_name,
            "Element #": match.group(1),
            "Element": match.group(2),
            "Base Value": float(match.group(3)),
            "GOE": float(match.group(4)),
            "Final Score": float(match.group(5))
        })

elements_df = pd.DataFrame(all_elements)

pcs_rows = []

for block in skater_blocks[1:]:
    
    # --- Extract skater name ---
    name_match = re.search(r"\d+\s+([A-Za-z\s\-']+)\s+[A-Z]{3}", block)
    if not name_match:
        continue
    
    skater_name = name_match.group(1).strip()
    
    # --- Extract PCS lines ---
    pcs_pattern = re.compile(
        r"(Composition|Presentation|Skating Skills)\s+([\d\.]+)\s+((?:\d+\.\d+\s+){8,9})(\d+\.\d+)"
    )
    
    for match in pcs_pattern.finditer(block):
        
        component = match.group(1)
        factor = float(match.group(2))
        
        # Judge scores
        judge_scores = match.group(3).strip().split()
        
        # Last value = panel average
        panel_avg = float(match.group(4))
        
        for i, score in enumerate(judge_scores):
            pcs_rows.append({
                "Skater": skater_name,
                "Component": component,
                "Judge": f"J{i+1}",
                "Score": float(score),
                "Factor": factor,
                "Panel Avg": panel_avg
            })

pcs_df = pd.DataFrame(pcs_rows)

pcs_wide = pcs_df.pivot_table(index=['Skater', 'Component'],columns='Judge',values='Score',sort = False)
panel_avg = pcs_df.groupby(['Skater', 'Component'])['Panel Avg'].first()
pcs_wide = pcs_wide.assign(Panel_Avg=panel_avg)
elements_wide = elements_df.pivot_table(index=['Skater','Element #','Element'],values=['Base Value','GOE','Final Score'])

# Turning them into csv files:
pcs_wide = pcs_wide.assign(Panel_Avg=panel_avg)
pcs_wide = pcs_wide.reset_index()
pcs_wide.to_csv("2024_AllenComp_MenFS_PCS.csv", index=False)

elements_wide = elements_df.pivot_table(
    index=['Skater', 'Element #', 'Element'],
    values=['Base Value', 'GOE', 'Final Score']
)

elements_wide = elements_wide.reset_index()

elements_wide.to_csv("2024_AllenComp_MenFS_Elements.csv", index=False)

In [20]:
# 2024 Allen Women FS

text = ""

with pdfplumber.open('2024_AllenComp_WomenFS.pdf') as pdf:
    for page in pdf.pages:
        text += page.extract_text() + "\n"

skater_blocks = text.split("Rank Name Nation")

all_elements = []

for block in skater_blocks[1:]:
    
    # --- Extract skater name ---
    name_match = re.search(r"\d+\s+([A-Za-z\s\-']+)\s+[A-Z]{3}", block)
    if not name_match:
        continue
    
    skater_name = name_match.group(1).strip()
    
    # --- Extract elements ---
    element_pattern = re.compile(
        r"\n\s*(\d+)\s+([A-Za-z0-9\+\*q<>]+)\s+([\d\.]+)\s+([\d\.\-]+).*?([\d\.]+)$",
        re.MULTILINE
    )
    
    for match in element_pattern.finditer(block):
        all_elements.append({
            "Skater": skater_name,
            "Element #": match.group(1),
            "Element": match.group(2),
            "Base Value": float(match.group(3)),
            "GOE": float(match.group(4)),
            "Final Score": float(match.group(5))
        })

elements_df = pd.DataFrame(all_elements)

pcs_rows = []

for block in skater_blocks[1:]:
    
    # --- Extract skater name ---
    name_match = re.search(r"\d+\s+([A-Za-z\s\-']+)\s+[A-Z]{3}", block)
    if not name_match:
        continue
    
    skater_name = name_match.group(1).strip()
    
    # --- Extract PCS lines ---
    pcs_pattern = re.compile(
        r"(Composition|Presentation|Skating Skills)\s+([\d\.]+)\s+((?:\d+\.\d+\s+){8,9})(\d+\.\d+)"
    )
    
    for match in pcs_pattern.finditer(block):
        
        component = match.group(1)
        factor = float(match.group(2))
        
        # Judge scores
        judge_scores = match.group(3).strip().split()
        
        # Last value = panel average
        panel_avg = float(match.group(4))
        
        for i, score in enumerate(judge_scores):
            pcs_rows.append({
                "Skater": skater_name,
                "Component": component,
                "Judge": f"J{i+1}",
                "Score": float(score),
                "Factor": factor,
                "Panel Avg": panel_avg
            })

pcs_df = pd.DataFrame(pcs_rows)

pcs_wide = pcs_df.pivot_table(index=['Skater', 'Component'],columns='Judge',values='Score',sort = False)
panel_avg = pcs_df.groupby(['Skater', 'Component'])['Panel Avg'].first()
pcs_wide = pcs_wide.assign(Panel_Avg=panel_avg)
elements_wide = elements_df.pivot_table(index=['Skater','Element #','Element'],values=['Base Value','GOE','Final Score'])

# Turning them into csv files:
pcs_wide = pcs_wide.assign(Panel_Avg=panel_avg)
pcs_wide = pcs_wide.reset_index()
pcs_wide.to_csv("2024_AllenComp_WomenFS_PCS.csv", index=False)

elements_wide = elements_df.pivot_table(
    index=['Skater', 'Element #', 'Element'],
    values=['Base Value', 'GOE', 'Final Score']
)

elements_wide = elements_wide.reset_index()

elements_wide.to_csv("2024_AllenComp_WomenFS_Elements.csv", index=False)

In [21]:
# 2024 Canada Mens FS

text = ""

with pdfplumber.open('2024_Canada_MensFS.pdf') as pdf:
    for page in pdf.pages:
        text += page.extract_text() + "\n"

skater_blocks = text.split("Rank Name Nation")

all_elements = []

for block in skater_blocks[1:]:
    
    # --- Extract skater name ---
    name_match = re.search(r"\d+\s+([A-Za-z\s\-']+)\s+[A-Z]{3}", block)
    if not name_match:
        continue
    
    skater_name = name_match.group(1).strip()
    
    # --- Extract elements ---
    element_pattern = re.compile(
        r"\n\s*(\d+)\s+([A-Za-z0-9\+\*q<>]+)\s+([\d\.]+)\s+([\d\.\-]+).*?([\d\.]+)$",
        re.MULTILINE
    )
    
    for match in element_pattern.finditer(block):
        all_elements.append({
            "Skater": skater_name,
            "Element #": match.group(1),
            "Element": match.group(2),
            "Base Value": float(match.group(3)),
            "GOE": float(match.group(4)),
            "Final Score": float(match.group(5))
        })

elements_df = pd.DataFrame(all_elements)

pcs_rows = []

for block in skater_blocks[1:]:
    
    # --- Extract skater name ---
    name_match = re.search(r"\d+\s+([A-Za-z\s\-']+)\s+[A-Z]{3}", block)
    if not name_match:
        continue
    
    skater_name = name_match.group(1).strip()
    
    # --- Extract PCS lines ---
    pcs_pattern = re.compile(
        r"(Composition|Presentation|Skating Skills)\s+([\d\.]+)\s+((?:\d+\.\d+\s+){8,9})(\d+\.\d+)"
    )
    
    for match in pcs_pattern.finditer(block):
        
        component = match.group(1)
        factor = float(match.group(2))
        
        # Judge scores
        judge_scores = match.group(3).strip().split()
        
        # Last value = panel average
        panel_avg = float(match.group(4))
        
        for i, score in enumerate(judge_scores):
            pcs_rows.append({
                "Skater": skater_name,
                "Component": component,
                "Judge": f"J{i+1}",
                "Score": float(score),
                "Factor": factor,
                "Panel Avg": panel_avg
            })

pcs_df = pd.DataFrame(pcs_rows)

pcs_wide = pcs_df.pivot_table(index=['Skater', 'Component'],columns='Judge',values='Score',sort = False)
panel_avg = pcs_df.groupby(['Skater', 'Component'])['Panel Avg'].first()
pcs_wide = pcs_wide.assign(Panel_Avg=panel_avg)
elements_wide = elements_df.pivot_table(index=['Skater','Element #','Element'],values=['Base Value','GOE','Final Score'])

# Turning them into csv files:
pcs_wide = pcs_wide.assign(Panel_Avg=panel_avg)
pcs_wide = pcs_wide.reset_index()
pcs_wide.to_csv("2024_Canada_MenFS_PCS.csv", index=False)

elements_wide = elements_df.pivot_table(
    index=['Skater', 'Element #', 'Element'],
    values=['Base Value', 'GOE', 'Final Score']
)

elements_wide = elements_wide.reset_index()

elements_wide.to_csv("2024_Canada_MenFS_Elements.csv", index=False)

In [22]:
# 2024 Canada Womens FS

text = ""

with pdfplumber.open('2024_Canada_WomenFS.pdf') as pdf:
    for page in pdf.pages:
        text += page.extract_text() + "\n"

skater_blocks = text.split("Rank Name Nation")

all_elements = []

for block in skater_blocks[1:]:
    
    # --- Extract skater name ---
    name_match = re.search(r"\d+\s+([A-Za-z\s\-']+)\s+[A-Z]{3}", block)
    if not name_match:
        continue
    
    skater_name = name_match.group(1).strip()
    
    # --- Extract elements ---
    element_pattern = re.compile(
        r"\n\s*(\d+)\s+([A-Za-z0-9\+\*q<>]+)\s+([\d\.]+)\s+([\d\.\-]+).*?([\d\.]+)$",
        re.MULTILINE
    )
    
    for match in element_pattern.finditer(block):
        all_elements.append({
            "Skater": skater_name,
            "Element #": match.group(1),
            "Element": match.group(2),
            "Base Value": float(match.group(3)),
            "GOE": float(match.group(4)),
            "Final Score": float(match.group(5))
        })

elements_df = pd.DataFrame(all_elements)

pcs_rows = []

for block in skater_blocks[1:]:
    
    # --- Extract skater name ---
    name_match = re.search(r"\d+\s+([A-Za-z\s\-']+)\s+[A-Z]{3}", block)
    if not name_match:
        continue
    
    skater_name = name_match.group(1).strip()
    
    # --- Extract PCS lines ---
    pcs_pattern = re.compile(
        r"(Composition|Presentation|Skating Skills)\s+([\d\.]+)\s+((?:\d+\.\d+\s+){8,9})(\d+\.\d+)"
    )
    
    for match in pcs_pattern.finditer(block):
        
        component = match.group(1)
        factor = float(match.group(2))
        
        # Judge scores
        judge_scores = match.group(3).strip().split()
        
        # Last value = panel average
        panel_avg = float(match.group(4))
        
        for i, score in enumerate(judge_scores):
            pcs_rows.append({
                "Skater": skater_name,
                "Component": component,
                "Judge": f"J{i+1}",
                "Score": float(score),
                "Factor": factor,
                "Panel Avg": panel_avg
            })

pcs_df = pd.DataFrame(pcs_rows)

pcs_wide = pcs_df.pivot_table(index=['Skater', 'Component'],columns='Judge',values='Score',sort = False)
panel_avg = pcs_df.groupby(['Skater', 'Component'])['Panel Avg'].first()
pcs_wide = pcs_wide.assign(Panel_Avg=panel_avg)
elements_wide = elements_df.pivot_table(index=['Skater','Element #','Element'],values=['Base Value','GOE','Final Score'])

# Turning them into csv files:
pcs_wide = pcs_wide.assign(Panel_Avg=panel_avg)
pcs_wide = pcs_wide.reset_index()
pcs_wide.to_csv("2024_Canada_WomenFS_PCS.csv", index=False)

elements_wide = elements_df.pivot_table(
    index=['Skater', 'Element #', 'Element'],
    values=['Base Value', 'GOE', 'Final Score']
)

elements_wide = elements_wide.reset_index()

elements_wide.to_csv("2024_Canada_WomenFS_Elements.csv", index=False)

In [23]:
# 2024 France Mens FS

text = ""

with pdfplumber.open('2024_France_MenFS.pdf') as pdf:
    for page in pdf.pages:
        text += page.extract_text() + "\n"

skater_blocks = text.split("Rank Name Nation")

all_elements = []

for block in skater_blocks[1:]:
    
    # --- Extract skater name ---
    name_match = re.search(r"\d+\s+([A-Za-z\s\-']+)\s+[A-Z]{3}", block)
    if not name_match:
        continue
    
    skater_name = name_match.group(1).strip()
    
    # --- Extract elements ---
    element_pattern = re.compile(
        r"\n\s*(\d+)\s+([A-Za-z0-9\+\*q<>]+)\s+([\d\.]+)\s+([\d\.\-]+).*?([\d\.]+)$",
        re.MULTILINE
    )
    
    for match in element_pattern.finditer(block):
        all_elements.append({
            "Skater": skater_name,
            "Element #": match.group(1),
            "Element": match.group(2),
            "Base Value": float(match.group(3)),
            "GOE": float(match.group(4)),
            "Final Score": float(match.group(5))
        })

elements_df = pd.DataFrame(all_elements)

pcs_rows = []

for block in skater_blocks[1:]:
    
    # --- Extract skater name ---
    name_match = re.search(r"\d+\s+([A-Za-z\s\-']+)\s+[A-Z]{3}", block)
    if not name_match:
        continue
    
    skater_name = name_match.group(1).strip()
    
    # --- Extract PCS lines ---
    pcs_pattern = re.compile(
        r"(Composition|Presentation|Skating Skills)\s+([\d\.]+)\s+((?:\d+\.\d+\s+){8,9})(\d+\.\d+)"
    )
    
    for match in pcs_pattern.finditer(block):
        
        component = match.group(1)
        factor = float(match.group(2))
        
        # Judge scores
        judge_scores = match.group(3).strip().split()
        
        # Last value = panel average
        panel_avg = float(match.group(4))
        
        for i, score in enumerate(judge_scores):
            pcs_rows.append({
                "Skater": skater_name,
                "Component": component,
                "Judge": f"J{i+1}",
                "Score": float(score),
                "Factor": factor,
                "Panel Avg": panel_avg
            })

pcs_df = pd.DataFrame(pcs_rows)

pcs_wide = pcs_df.pivot_table(index=['Skater', 'Component'],columns='Judge',values='Score',sort = False)
panel_avg = pcs_df.groupby(['Skater', 'Component'])['Panel Avg'].first()
pcs_wide = pcs_wide.assign(Panel_Avg=panel_avg)
elements_wide = elements_df.pivot_table(index=['Skater','Element #','Element'],values=['Base Value','GOE','Final Score'])

# Turning them into csv files:
pcs_wide = pcs_wide.assign(Panel_Avg=panel_avg)
pcs_wide = pcs_wide.reset_index()
pcs_wide.to_csv("2024_France_MenFS_PCS.csv", index=False)

elements_wide = elements_df.pivot_table(
    index=['Skater', 'Element #', 'Element'],
    values=['Base Value', 'GOE', 'Final Score']
)

elements_wide = elements_wide.reset_index()

elements_wide.to_csv("2024_France_MenFS_Elements.csv", index=False)

In [24]:
# 2024 France Women FS

text = ""

with pdfplumber.open('2024_France_WomenFS.pdf') as pdf:
    for page in pdf.pages:
        text += page.extract_text() + "\n"

skater_blocks = text.split("Rank Name Nation")

all_elements = []

for block in skater_blocks[1:]:
    
    # --- Extract skater name ---
    name_match = re.search(r"\d+\s+([A-Za-z\s\-']+)\s+[A-Z]{3}", block)
    if not name_match:
        continue
    
    skater_name = name_match.group(1).strip()
    
    # --- Extract elements ---
    element_pattern = re.compile(
        r"\n\s*(\d+)\s+([A-Za-z0-9\+\*q<>]+)\s+([\d\.]+)\s+([\d\.\-]+).*?([\d\.]+)$",
        re.MULTILINE
    )
    
    for match in element_pattern.finditer(block):
        all_elements.append({
            "Skater": skater_name,
            "Element #": match.group(1),
            "Element": match.group(2),
            "Base Value": float(match.group(3)),
            "GOE": float(match.group(4)),
            "Final Score": float(match.group(5))
        })

elements_df = pd.DataFrame(all_elements)

pcs_rows = []

for block in skater_blocks[1:]:
    
    # --- Extract skater name ---
    name_match = re.search(r"\d+\s+([A-Za-z\s\-']+)\s+[A-Z]{3}", block)
    if not name_match:
        continue
    
    skater_name = name_match.group(1).strip()
    
    # --- Extract PCS lines ---
    pcs_pattern = re.compile(
        r"(Composition|Presentation|Skating Skills)\s+([\d\.]+)\s+((?:\d+\.\d+\s+){8,9})(\d+\.\d+)"
    )
    
    for match in pcs_pattern.finditer(block):
        
        component = match.group(1)
        factor = float(match.group(2))
        
        # Judge scores
        judge_scores = match.group(3).strip().split()
        
        # Last value = panel average
        panel_avg = float(match.group(4))
        
        for i, score in enumerate(judge_scores):
            pcs_rows.append({
                "Skater": skater_name,
                "Component": component,
                "Judge": f"J{i+1}",
                "Score": float(score),
                "Factor": factor,
                "Panel Avg": panel_avg
            })

pcs_df = pd.DataFrame(pcs_rows)

pcs_wide = pcs_df.pivot_table(index=['Skater', 'Component'],columns='Judge',values='Score',sort = False)
panel_avg = pcs_df.groupby(['Skater', 'Component'])['Panel Avg'].first()
pcs_wide = pcs_wide.assign(Panel_Avg=panel_avg)
elements_wide = elements_df.pivot_table(index=['Skater','Element #','Element'],values=['Base Value','GOE','Final Score'])

# Turning them into csv files:
pcs_wide = pcs_wide.assign(Panel_Avg=panel_avg)
pcs_wide = pcs_wide.reset_index()
pcs_wide.to_csv("2024_France_WomenFS_PCS.csv", index=False)

elements_wide = elements_df.pivot_table(
    index=['Skater', 'Element #', 'Element'],
    values=['Base Value', 'GOE', 'Final Score']
)

elements_wide = elements_wide.reset_index()

elements_wide.to_csv("2024_France_WomenFS_Elements.csv", index=False)

In [25]:
# 2024 GPFinal Men FS

text = ""

with pdfplumber.open('2024_GPFinal_MenFS.pdf') as pdf:
    for page in pdf.pages:
        text += page.extract_text() + "\n"

skater_blocks = text.split("Rank Name Nation")

all_elements = []

for block in skater_blocks[1:]:
    
    # --- Extract skater name ---
    name_match = re.search(r"\d+\s+([A-Za-z\s\-']+)\s+[A-Z]{3}", block)
    if not name_match:
        continue
    
    skater_name = name_match.group(1).strip()
    
    # --- Extract elements ---
    element_pattern = re.compile(
        r"\n\s*(\d+)\s+([A-Za-z0-9\+\*q<>]+)\s+([\d\.]+)\s+([\d\.\-]+).*?([\d\.]+)$",
        re.MULTILINE
    )
    
    for match in element_pattern.finditer(block):
        all_elements.append({
            "Skater": skater_name,
            "Element #": match.group(1),
            "Element": match.group(2),
            "Base Value": float(match.group(3)),
            "GOE": float(match.group(4)),
            "Final Score": float(match.group(5))
        })

elements_df = pd.DataFrame(all_elements)

pcs_rows = []

for block in skater_blocks[1:]:
    
    # --- Extract skater name ---
    name_match = re.search(r"\d+\s+([A-Za-z\s\-']+)\s+[A-Z]{3}", block)
    if not name_match:
        continue
    
    skater_name = name_match.group(1).strip()
    
    # --- Extract PCS lines ---
    pcs_pattern = re.compile(
        r"(Composition|Presentation|Skating Skills)\s+([\d\.]+)\s+((?:\d+\.\d+\s+){8,9})(\d+\.\d+)"
    )
    
    for match in pcs_pattern.finditer(block):
        
        component = match.group(1)
        factor = float(match.group(2))
        
        # Judge scores
        judge_scores = match.group(3).strip().split()
        
        # Last value = panel average
        panel_avg = float(match.group(4))
        
        for i, score in enumerate(judge_scores):
            pcs_rows.append({
                "Skater": skater_name,
                "Component": component,
                "Judge": f"J{i+1}",
                "Score": float(score),
                "Factor": factor,
                "Panel Avg": panel_avg
            })

pcs_df = pd.DataFrame(pcs_rows)

pcs_wide = pcs_df.pivot_table(index=['Skater', 'Component'],columns='Judge',values='Score',sort = False)
panel_avg = pcs_df.groupby(['Skater', 'Component'])['Panel Avg'].first()
pcs_wide = pcs_wide.assign(Panel_Avg=panel_avg)
elements_wide = elements_df.pivot_table(index=['Skater','Element #','Element'],values=['Base Value','GOE','Final Score'])

# Turning them into csv files:
pcs_wide = pcs_wide.assign(Panel_Avg=panel_avg)
pcs_wide = pcs_wide.reset_index()
pcs_wide.to_csv("2024_GPFinal_MenFS_PCS.csv", index=False)

elements_wide = elements_df.pivot_table(
    index=['Skater', 'Element #', 'Element'],
    values=['Base Value', 'GOE', 'Final Score']
)

elements_wide = elements_wide.reset_index()

elements_wide.to_csv("2024_GPFinal_MenFS_Elements.csv", index=False)

In [26]:
# 2024 GPFinal Women FS

text = ""

with pdfplumber.open('2024_GPFinal_WomenFS.pdf') as pdf:
    for page in pdf.pages:
        text += page.extract_text() + "\n"

skater_blocks = text.split("Rank Name Nation")

all_elements = []

for block in skater_blocks[1:]:
    
    # --- Extract skater name ---
    name_match = re.search(r"\d+\s+([A-Za-z\s\-']+)\s+[A-Z]{3}", block)
    if not name_match:
        continue
    
    skater_name = name_match.group(1).strip()
    
    # --- Extract elements ---
    element_pattern = re.compile(
        r"\n\s*(\d+)\s+([A-Za-z0-9\+\*q<>]+)\s+([\d\.]+)\s+([\d\.\-]+).*?([\d\.]+)$",
        re.MULTILINE
    )
    
    for match in element_pattern.finditer(block):
        all_elements.append({
            "Skater": skater_name,
            "Element #": match.group(1),
            "Element": match.group(2),
            "Base Value": float(match.group(3)),
            "GOE": float(match.group(4)),
            "Final Score": float(match.group(5))
        })

elements_df = pd.DataFrame(all_elements)

pcs_rows = []

for block in skater_blocks[1:]:
    
    # --- Extract skater name ---
    name_match = re.search(r"\d+\s+([A-Za-z\s\-']+)\s+[A-Z]{3}", block)
    if not name_match:
        continue
    
    skater_name = name_match.group(1).strip()
    
    # --- Extract PCS lines ---
    pcs_pattern = re.compile(
        r"(Composition|Presentation|Skating Skills)\s+([\d\.]+)\s+((?:\d+\.\d+\s+){8,9})(\d+\.\d+)"
    )
    
    for match in pcs_pattern.finditer(block):
        
        component = match.group(1)
        factor = float(match.group(2))
        
        # Judge scores
        judge_scores = match.group(3).strip().split()
        
        # Last value = panel average
        panel_avg = float(match.group(4))
        
        for i, score in enumerate(judge_scores):
            pcs_rows.append({
                "Skater": skater_name,
                "Component": component,
                "Judge": f"J{i+1}",
                "Score": float(score),
                "Factor": factor,
                "Panel Avg": panel_avg
            })

pcs_df = pd.DataFrame(pcs_rows)

pcs_wide = pcs_df.pivot_table(index=['Skater', 'Component'],columns='Judge',values='Score',sort = False)
panel_avg = pcs_df.groupby(['Skater', 'Component'])['Panel Avg'].first()
pcs_wide = pcs_wide.assign(Panel_Avg=panel_avg)
elements_wide = elements_df.pivot_table(index=['Skater','Element #','Element'],values=['Base Value','GOE','Final Score'])

# Turning them into csv files:
pcs_wide = pcs_wide.assign(Panel_Avg=panel_avg)
pcs_wide = pcs_wide.reset_index()
pcs_wide.to_csv("2024_GPFinal_WomenFS_PCS.csv", index=False)

elements_wide = elements_df.pivot_table(
    index=['Skater', 'Element #', 'Element'],
    values=['Base Value', 'GOE', 'Final Score']
)

elements_wide = elements_wide.reset_index()

elements_wide.to_csv("2024_GPFinal_WomenFS_Elements.csv", index=False)

In [27]:
# 2024 Japan Men FS

text = ""

with pdfplumber.open('2024_Japan_MenFS.pdf') as pdf:
    for page in pdf.pages:
        text += page.extract_text() + "\n"

skater_blocks = text.split("Rank Name Nation")

all_elements = []

for block in skater_blocks[1:]:
    
    # --- Extract skater name ---
    name_match = re.search(r"\d+\s+([A-Za-z\s\-']+)\s+[A-Z]{3}", block)
    if not name_match:
        continue
    
    skater_name = name_match.group(1).strip()
    
    # --- Extract elements ---
    element_pattern = re.compile(
        r"\n\s*(\d+)\s+([A-Za-z0-9\+\*q<>]+)\s+([\d\.]+)\s+([\d\.\-]+).*?([\d\.]+)$",
        re.MULTILINE
    )
    
    for match in element_pattern.finditer(block):
        all_elements.append({
            "Skater": skater_name,
            "Element #": match.group(1),
            "Element": match.group(2),
            "Base Value": float(match.group(3)),
            "GOE": float(match.group(4)),
            "Final Score": float(match.group(5))
        })

elements_df = pd.DataFrame(all_elements)

pcs_rows = []

for block in skater_blocks[1:]:
    
    # --- Extract skater name ---
    name_match = re.search(r"\d+\s+([A-Za-z\s\-']+)\s+[A-Z]{3}", block)
    if not name_match:
        continue
    
    skater_name = name_match.group(1).strip()
    
    # --- Extract PCS lines ---
    pcs_pattern = re.compile(
        r"(Composition|Presentation|Skating Skills)\s+([\d\.]+)\s+((?:\d+\.\d+\s+){8,9})(\d+\.\d+)"
    )
    
    for match in pcs_pattern.finditer(block):
        
        component = match.group(1)
        factor = float(match.group(2))
        
        # Judge scores
        judge_scores = match.group(3).strip().split()
        
        # Last value = panel average
        panel_avg = float(match.group(4))
        
        for i, score in enumerate(judge_scores):
            pcs_rows.append({
                "Skater": skater_name,
                "Component": component,
                "Judge": f"J{i+1}",
                "Score": float(score),
                "Factor": factor,
                "Panel Avg": panel_avg
            })

pcs_df = pd.DataFrame(pcs_rows)

pcs_wide = pcs_df.pivot_table(index=['Skater', 'Component'],columns='Judge',values='Score',sort = False)
panel_avg = pcs_df.groupby(['Skater', 'Component'])['Panel Avg'].first()
pcs_wide = pcs_wide.assign(Panel_Avg=panel_avg)
elements_wide = elements_df.pivot_table(index=['Skater','Element #','Element'],values=['Base Value','GOE','Final Score'])

# Turning them into csv files:
pcs_wide = pcs_wide.assign(Panel_Avg=panel_avg)
pcs_wide = pcs_wide.reset_index()
pcs_wide.to_csv("2024_Japan_MenFS_PCS.csv", index=False)

elements_wide = elements_df.pivot_table(
    index=['Skater', 'Element #', 'Element'],
    values=['Base Value', 'GOE', 'Final Score']
)

elements_wide = elements_wide.reset_index()

elements_wide.to_csv("2024_Japan_MenFS_Elements.csv", index=False)

In [28]:
# 2024 Japan Women FS

text = ""

with pdfplumber.open('2024_Japan_WomenFS.pdf') as pdf:
    for page in pdf.pages:
        text += page.extract_text() + "\n"

skater_blocks = text.split("Rank Name Nation")

all_elements = []

for block in skater_blocks[1:]:
    
    # --- Extract skater name ---
    name_match = re.search(r"\d+\s+([A-Za-z\s\-']+)\s+[A-Z]{3}", block)
    if not name_match:
        continue
    
    skater_name = name_match.group(1).strip()
    
    # --- Extract elements ---
    element_pattern = re.compile(
        r"\n\s*(\d+)\s+([A-Za-z0-9\+\*q<>]+)\s+([\d\.]+)\s+([\d\.\-]+).*?([\d\.]+)$",
        re.MULTILINE
    )
    
    for match in element_pattern.finditer(block):
        all_elements.append({
            "Skater": skater_name,
            "Element #": match.group(1),
            "Element": match.group(2),
            "Base Value": float(match.group(3)),
            "GOE": float(match.group(4)),
            "Final Score": float(match.group(5))
        })

elements_df = pd.DataFrame(all_elements)

pcs_rows = []

for block in skater_blocks[1:]:
    
    # --- Extract skater name ---
    name_match = re.search(r"\d+\s+([A-Za-z\s\-']+)\s+[A-Z]{3}", block)
    if not name_match:
        continue
    
    skater_name = name_match.group(1).strip()
    
    # --- Extract PCS lines ---
    pcs_pattern = re.compile(
        r"(Composition|Presentation|Skating Skills)\s+([\d\.]+)\s+((?:\d+\.\d+\s+){8,9})(\d+\.\d+)"
    )
    
    for match in pcs_pattern.finditer(block):
        
        component = match.group(1)
        factor = float(match.group(2))
        
        # Judge scores
        judge_scores = match.group(3).strip().split()
        
        # Last value = panel average
        panel_avg = float(match.group(4))
        
        for i, score in enumerate(judge_scores):
            pcs_rows.append({
                "Skater": skater_name,
                "Component": component,
                "Judge": f"J{i+1}",
                "Score": float(score),
                "Factor": factor,
                "Panel Avg": panel_avg
            })

pcs_df = pd.DataFrame(pcs_rows)

pcs_wide = pcs_df.pivot_table(index=['Skater', 'Component'],columns='Judge',values='Score',sort = False)
panel_avg = pcs_df.groupby(['Skater', 'Component'])['Panel Avg'].first()
pcs_wide = pcs_wide.assign(Panel_Avg=panel_avg)
elements_wide = elements_df.pivot_table(index=['Skater','Element #','Element'],values=['Base Value','GOE','Final Score'])

# Turning them into csv files:
pcs_wide = pcs_wide.assign(Panel_Avg=panel_avg)
pcs_wide = pcs_wide.reset_index()
pcs_wide.to_csv("2024_Japan_WomenFS_PCS.csv", index=False)

elements_wide = elements_df.pivot_table(
    index=['Skater', 'Element #', 'Element'],
    values=['Base Value', 'GOE', 'Final Score']
)

elements_wide = elements_wide.reset_index()

elements_wide.to_csv("2024_Japan_WomenFS_Elements.csv", index=False)